### RAG pipelines- Data Ingestion to vector DB pipeline

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\chand\AppData\Local\Temp\ipykernel_18712\885881214.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [3]:
from pathlib import Path
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf_files")

Found 2 PDF files to process

Processing: combinatorial.pdf
  ✓ Loaded 2 pages

Processing: ethics.pdf
  ✓ Loaded 2 pages

Total documents loaded: 4


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}, page_content='CSE357\n:\nCOMBINATORIAL STUDIES\nCourse Outcomes:\nCO1 :: \nunderstand the fundamental computer science concepts, including data structures,  \nalgorithms, databases, operating systems, and computer networks, essential for technical  \ninterviews.\nCO2 :: \nassess the problem-solving skills specific to coding challenges and algorithmic problems  \nfrequently encountered in technical interviews.\nCO3 :: \narticulate a comprehensive command of Object-Oriented Programming principles,  \nenhancing readiness to excel in technical interviews.\nUnit I\nOperating System

In [5]:
### text splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.create_documents([doc.page_content for doc in documents], metadatas=[doc.metadata for doc in documents])
    print(f"Split into {len(split_docs)} chunks")
    if split_docs:
        print(f"Example chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs

In [6]:
chunks=split_documents(all_pdf_documents)
chunks

Split into 10 chunks
Example chunk:
Content: CSE357
:
COMBINATORIAL STUDIES
Course Outcomes:
CO1 :: 
understand the fundamental computer science concepts, including data structures,  
algorithms, databases, operating systems, and computer networ...
Metadata: {'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}, page_content='CSE357\n:\nCOMBINATORIAL STUDIES\nCourse Outcomes:\nCO1 :: \nunderstand the fundamental computer science concepts, including data structures,  \nalgorithms, databases, operating systems, and computer networks, essential for technical  \ninterviews.\nCO2 :: \nassess the problem-solving skills specific to coding challenges and algorithmic problems  \nfrequently encountered in technical interviews.\nCO3 :: \narticulate a comprehensive command of Object-Oriented Programming principles,  \nenhancing readiness to excel in technical interviews.\nUnit I\nOperating System

### emabeding and vectorstore DB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class embeddingManager:
    """handle document embedding generation using sentence transformers
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager
        Args:
            model_name (str, optional): The name of the sentence transformer model to use. Defaults to "all-MiniLM-L6-v2".
        """
        self.model_name = model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model
        """
        try:
            print(f"Loading model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully.:{self.model.get_sentence_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        Args:
            texts (List[str]): A list of strings to generate embeddings for
        Returns:
            np.ndarray: A 2D array of shape (len(texts), embedding_dimension) containing the embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Call _load_model() first.")
     
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
      
## initialize embedding manager
embedding_manager = embeddingManager()
embedding_manager


Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2798.85it/s]


Model loaded successfully.:384 dimensions


C:\Users\chand\AppData\Local\Temp\ipykernel_18712\2946198722.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully.:{self.model.get_sentence_embedding_dimension()} dimensions")


### vector store

In [9]:
class VectorStore:
    """
    A simple vector store using ChromaDB to store document embeddings and metadata.
    """

    def __init__(
        self,
        collection_name: str = "pdf_chunks",
        persist_directory: str = "../data/vector_store",
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory where ChromaDB data is stored.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """
        Initialize the ChromaDB client and collection.
        """
        try:
            # Create persistence directory
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"hnsw:space": "cosine"},
            )

            print(
                f"Vector store initialized with collection: "
                f"{self.collection_name}"
            )
            print(
                f"Current number of documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray,
    ):
        """
        Add documents and embeddings to ChromaDB.

        Args:
            documents: List of LangChain documents.
            embeddings: Numpy array of embeddings.
        """
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):
            # Unique document ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # ChromaDB metadata must contain primitive values
            metadata = {
                k: (
                    v
                    if isinstance(v, (str, int, float, bool))
                    else str(v)
                )
                for k, v in doc.metadata.items()
            }

            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)

            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
            )

            print(
                f"Successfully added {len(documents)} documents"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# Create vector store
vectorstore = VectorStore()
print(vectorstore)

Vector store initialized with collection: pdf_chunks
Current number of documents in collection: 40


In [10]:
chunks

[Document(metadata={'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0', 'creator': 'Microsoft Reporting Services 2022.1.0.0', 'creationdate': '2026-06-07T12:47:24+05:30', 'title': 'rdlCourseSyllabusNew', 'author': '', 'subject': '', 'source': '..\\data\\pdf_files\\combinatorial.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'combinatorial.pdf', 'file_type': 'pdf'}, page_content='CSE357\n:\nCOMBINATORIAL STUDIES\nCourse Outcomes:\nCO1 :: \nunderstand the fundamental computer science concepts, including data structures,  \nalgorithms, databases, operating systems, and computer networks, essential for technical  \ninterviews.\nCO2 :: \nassess the problem-solving skills specific to coding challenges and algorithmic problems  \nfrequently encountered in technical interviews.\nCO3 :: \narticulate a comprehensive command of Object-Oriented Programming principles,  \nenhancing readiness to excel in technical interviews.\nUnit I\nOperating System

In [11]:
### convert the text to embeddings
text=[doc.page_content for doc in chunks]
text
### generate embeddings
embeddings=embedding_manager.generate_embeddings(text)

### add to vector store
vectorstore.add_documents(chunks,embeddings)


Generating embeddings for 10 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]

Generated embeddings with shape: (10, 384)
Adding 10 documents to vector store...
Successfully added 10 documents
Total documents in collection: 50


### Retriever pipeline from vectorStore


In [12]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: embeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [13]:
rag_retriever

In [14]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.97it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_95d8a1b5_3',
  'content': 'simulations of behavioural interview scenarios, fostering confidence and preparedness for  \ndifferent stages of technical interviews.\nThrough this course students should be able to\nL:\n2\n   \nT:\n0\n   \nP:\n2\n    \nCredits\n:\n3\n  \nList of Practicals\n•\n1. Introduction to Linux Commands\n•\n2. Shell Programming\n•\n3. System Calls (File / Directory/ Process)\n•\n4. Multi Thread Process using Pthread Library\n•\n5. SQL Commands ( DDL/DML/ TCL)\nList of Practicals / Experiments:\nSession  20\n25\n-\n26\n                  Page:\n1\n/\n2',
  'metadata': {'page_label': '1',
   'page': 0,
   'source': '..\\data\\pdf_files\\combinatorial.pdf',
   'producer': 'Microsoft Reporting Services PDF Rendering Extension 2022.1.0.0',
   'title': 'rdlCourseSyllabusNew',
   'doc_index': 3,
   'content_length': 513,
   'total_pages': 2,
   'subject': '',
   'source_file': 'combinatorial.pdf',
   'creationdate': '2026-06-07T12:47:24+05:30',
   'author': '',


### Integration VectorDB context pipeline with LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retrieve the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt_template="""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke(prompt_template.format(context=context,query=query))
    return response.content

c:\Users\chand\OneDrive\Desktop\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
answer =rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


There is no information about the attention mechanism in the provided context. The context appears to be related to a course syllabus, focusing on technical interviews, Linux commands, shell programming, system calls, and SQL commands.


### Enhanced RAG Pipeline Features 

In [17]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("what is SQL", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'what is SQL'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.17it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: SQL (Structured Query Language) is a programming language designed for managing and manipulating data in relational database management systems.
Sources: [{'source': 'combinatorial.pdf', 'page': 1, 'score': 0.3844485282897949, 'preview': 'Text\n Books\n:\n1\n. \nIT INTERVIEW QUESTIONS by NARASIMHA KARUMANCHI, CAREERMONK PUBLICATIONS \nReference\ns\n:\n1\n. \nMCQS IN COMPUTER SCIENCE by TIMOTHY WILLIAMS, MC GRAW HILL \n2\n. \nSQL IN 10 MINUTES, SAMS TEACH YOURSELF by BEN FORTA, SAMS PUBLISHING \n3\n. \nCRACKING THE IT INTERVIEW by M BALASUBRAMANIAM, K...'}, {'source': 'combinatorial.pdf', 'page': 1, 'score': 0.3844485282897949, 'preview': 'Text\n Books\n:\n1\n. \nIT INTERVIEW QUESTIONS by NARASIMHA KARUMANCHI, CAREERMONK PUBLICATIONS \nReference\ns\n:\n1\n. \nMCQS IN COMPUTER SCIENCE by TIMOTHY WILLIAMS, MC GRAW HILL \n2\n. \nSQL IN 10 MINUTES, SAMS TEACH YOURSELF by BEN FORTA, SAMS PUBLISHING \n3\n. \nCRACKING THE IT INTERVIEW by M BALASUBRAMANIAM, K...'}, {'source': 'combinato